# RAG-Based Internship Matching System Demo
### Infosys Springboard Virtual Internship Program 7.0

Welcome to the interactive demonstration of the **RAG-Based Internship Matching System**!

This notebook demonstrates how semantic search and vector embeddings can match candidate resumes against internship opportunities using:
- **Resume Preprocessing**: Normalizing structured candidate JSON
- **Vector Embeddings**: Sentence Transformers (`all-MiniLM-L6-v2` / 384 dimensions)
- **Vector Database**: FAISS (Inner Product / Normalized Cosine Similarity)
- **RAG Grounded Explanations**: Generating honest, anti-hallucinatory explanations

## 1. Environment Setup and Imports

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.config import settings
from app.services.resume_preprocessor import ResumePreprocessor
from app.services.internship_loader import InternshipLoader
from app.services.embedding_service import embedding_service
from app.services.vector_store import vector_store
from app.services.matching_service import matching_service
from app.services.explanation_service import explanation_service

print(f"Project Root: {PROJECT_ROOT}")
print(f"Embedding Model: {settings.embedding_model}")
print(f"Embedding Dimension: {settings.embedding_dimension}")

## 2. Load Internship Dataset
We load the synthetic internship opportunities from `data/internships.json`.

In [ ]:
loader = InternshipLoader(settings.internships_file)
internships = loader.load_dataset()
print(f"Loaded {len(internships)} internships.")

# Display sample internship
sample_internship = internships[0]
print(json.dumps(sample_internship.model_dump(), indent=2))

## 3. Resume Preprocessing Pipeline
We take candidate resume JSON (derived from the previous Resume Parsing milestone) and convert it into a rich textual representation.

In [ ]:
sample_candidate = {
    "name": "Arun",
    "summary": "Computer Science student interested in backend development and microservices.",
    "skills": ["Java", "Spring Boot", "SQL", "REST APIs", "PostgreSQL"],
    "education": ["B.Tech Computer Science and Engineering"],
    "projects": [
        "Developed an E-Commerce REST API using Spring Boot, Hibernate, and PostgreSQL",
        "Implemented microservices architecture with JWT authentication"
    ],
    "experience": ["Backend Intern at Tech Club"],
    "certifications": ["Java Full Stack Developer"]
}

preprocessed_text = ResumePreprocessor.preprocess_to_text(sample_candidate)
print("Preprocessed Candidate Text Representation:")
print(preprocessed_text)

## 4. Vector Embedding Generation
Generate normalized 384-dimensional dense vectors for both candidate and internship representations.

In [ ]:
candidate_vector = embedding_service.encode_text(preprocessed_text)
print(f"Candidate Vector Shape: {candidate_vector.shape}")
print(f"Vector L2 Norm: {np.linalg.norm(candidate_vector):.4f}")
print(f"First 10 Embedding Dimensions:")
print(candidate_vector[:10])

## 5. Verify / Load FAISS Vector Store
Ensure the FAISS index is loaded into memory.

In [ ]:
if not vector_store.is_loaded():
    vector_store.load_vector_index()

print(f"FAISS Vector Store Loaded: {vector_store.is_loaded()}")
print(f"Total Indexed Internships: {vector_store.get_total_indexed()}")

## 6. Perform Semantic Matching (Top-5 Retrieval)
Search for the top 5 most similar internships based on cosine similarity.

In [ ]:
match_results = matching_service.find_matches(
    candidate_data=sample_candidate,
    top_k=5,
    include_explanation=True
)

print(f"Matches found for: {match_results.candidate_name}")
print(f"Total Matches: {match_results.total_matches}\n")

results_table = []
for idx, m in enumerate(match_results.matches, start=1):
    results_table.append({
        "Rank": idx,
        "ID": m.internship_id,
        "Title": m.title,
        "Company": m.company,
        "Similarity": f"{m.similarity_score:.4f}",
        "Work Mode": m.work_mode,
        "Location": m.location
    })

df_results = pd.DataFrame(results_table)
df_results

## 7. Inspect Grounded RAG Explanations
Review the grounded matching rationale generated for the #1 match.

In [ ]:
top_match = match_results.matches[0]
expl = top_match.explanation

print(f"--- Explanation for Top Match: {top_match.title} at {top_match.company} ---")
print(f"Why Matched: {expl.match_reason}\n")
print(f"Matching Skills: {expl.matching_skills}")
print(f"Missing Skills: {expl.missing_required_skills}")
print(f"Relevant Projects: {expl.relevant_projects}\n")
print(f"Actionable Recommendation: {expl.recommendation}\n")
print(f"Disclaimer: {expl.disclaimer}")

## 8. Multi-Candidate Scenario Testing
Now let's test multiple distinct candidates across different technical domains:
1. Data Science Candidate
2. Cloud / DevOps Candidate
3. Cybersecurity Candidate

In [ ]:
with open(settings.test_candidates_file, "r", encoding="utf-8") as f:
    all_test_candidates = json.load(f)

selected_scenarios = ["Data Science", "Cloud Computing", "Cybersecurity"]

for cand in all_test_candidates:
    if cand["scenario"] in selected_scenarios:
        print(f"\n==================== Scenario: {cand['scenario']} ====================")
        res = matching_service.find_matches(cand["resume"], top_k=3)
        print(f"Candidate: {cand['resume']['name']}")
        for rank, m in enumerate(res.matches, 1):
            print(f"  #{rank} [{m.similarity_score:.4f}] {m.title} at {m.company} ({m.location})")

## 9. Conclusion & Key Takeaways
- **Semantic Proximity**: Dense vectors retrieve role titles that match concepts and project semantics, not just exact keyword strings.
- **Anti-Hallucination**: The explanation layer strictly cites supplied resume skills and explicitly flags missing prerequisites.
- **Fast Execution**: FAISS inner product search delivers sub-millisecond retrieval across indexed opportunities.